# Lab 41 (solution): Operating the maintenance loop

Reference implementation. The three pieces that make the [Lab 38](../../38-calibrating-the-eval-gate/) / [Lab 39](../../39-router-data-lifecycle/) machinery run on a cadence around its human gates: a **drift trigger** on live traffic, a **notifier** off the nightly monitor, and a **scheduled loop** with an explicit human review gate.

Ships `drift_check.py`, `notify.py`, `run_loop.py`, a sample window (`recent_queries.example.jsonl`), and three workflows.

## Step 0: Setup

In [ ]:
import json
import pathlib
import statistics
# This lab is operations: the three pieces that turn the Lab 38/39 machinery into a
# system that runs itself around its human gates. The pure decision logic lives in the
# scripts shipped here (drift_check.py, notify.py, run_loop.py); the cells import and
# exercise it, then point at the workflows that schedule it.
import sys
sys.path.insert(0, str(pathlib.Path.cwd()))
print("ops lab: drift trigger -> notifier -> scheduled loop")

## Step 1: The drift trigger (item 2)

Label-free confidence monitor on recent queries.

In [ ]:
from drift_check import drift_status
# Item 2: the trigger. Router confidence on a window of recent queries, compared to a
# baseline band. A sustained sag = phrasing the trainset never saw = start a new round.
healthy=[0.81,0.78,0.83,0.79,0.80]
drifted=[0.45,0.40,0.38,0.50,0.42]   # what messy live traffic looks like (Lab 39)
print("healthy window:", drift_status(healthy))
print("drifted window:", drift_status(drifted))
print("\nrag-drift-check.yml runs this weekly; exit code 2 (retrain_due) opens an issue.")
print("No labels needed - confidence is a label-free signal you can watch continuously.")

## Step 2: The notifier (item 3)

A regression becomes an alert.

In [ ]:
from notify import should_notify, format_alert, post
# Item 3: the notifier. The nightly faithfulness job writes faithfulness.json; this turns
# a regression into an alert. A summary nobody reads is not an alert.
print("regression -> notify?", should_notify(0.70, 0.764))
print("within band -> notify?", should_notify(0.80, 0.764))
alert=format_alert("judged_faithfulness", 0.70, 0.764, run_url="https://…/run/42")
print("\npayload:", json.dumps(alert))
print(post(alert, webhook=None))   # safe no-op without a webhook configured
print("\nThe nightly workflow runs this with ALERT_WEBHOOK_URL from secrets; unset = no-op.")

## Step 3: The loop orchestration (item 4)

Triage split + promotion gate, the two phases with a human between.

In [ ]:
from run_loop import triage_split, should_promote
# Item 4: the loop. A human-judgment step in the middle means you automate AROUND the
# human, not the human away. Two phases with a gate between.
items=[({"q":"helix lab boss??"},0.41),({"q":"what is cosine sim"},0.88),({"q":"who funds cascade??"},0.39)]
auto,review=triage_split(items, threshold=0.50)
print(f"prepare phase: {len(auto)} auto-accept, {len(review)} -> human review queue")

# promote gate: ship the retrained model only on a measured lift past a margin.
print("\npromote gate (min_lift=0.02):")
for a,b in [(0.80,0.86),(0.80,0.81),(0.80,0.78)]:
    print(f"  A={a} B={b} -> promote? {should_promote(a,b,0.02)}")
print("\nA tiny or negative delta does NOT promote - that is the guard against shipping on noise.")

## Step 4: The cadence and the human gates

In [ ]:
# The cadence (all in .github/workflows/):
#   rag-drift-check.yml        weekly  -> confidence drift -> opens a retrain-due issue
#   rag-maintenance-loop.yml   weekly  -> PREPARE: capture->dedup->triage->review queue (artifact)
#                              manual  -> PROMOTE: retrain->measure->gate->re-baseline (after review)
#   rag-faithfulness-nightly   nightly -> judged faithfulness -> notify.py on regression
#
# Human gates are explicit and load-bearing:
#   - PROMOTE is workflow_dispatch only: a person confirms the measured lift before shipping.
#   - the review queue is an artifact a person labels; the loop does not invent labels.
print("Automate the deterministic steps on a schedule; keep humans on the judgment steps.")

## Step 5: What this buys you

In [ ]:
# The maturity move this lab makes: the system now tells YOU when to act.
#  - drift check: confidence sags -> issue (start a round).
#  - nightly + notify: faithfulness regresses -> alert (investigate).
#  - maintenance loop: schedule the busywork, gate the decisions on a human + a measured lift.
# After any promote, re-run Lab 38's derive_thresholds.py - the model changed, so the
# baseline did too.
print("A maintained system is not one that never drifts - it is one that tells you when")
print("it has, and makes the fix a routine you already wrote down.")

## What you built

The operations layer: `drift_check.py` (+ `rag-drift-check.yml`) watches router confidence on live traffic and opens a retrain-due issue when it sags; `notify.py` (wired into the nightly faithfulness workflow) turns a regression into an alert; `run_loop.py` (+ `rag-maintenance-loop.yml`) schedules the deterministic front half of the retraining loop and gates the promote half on a human review and a measured lift. The recurring stance: **automate the deterministic steps, keep humans on the judgment steps.**

**Where this simplifies:** the drift baseline is a recorded constant (re-record it after each retrain); the notifier targets a generic webhook (map it to your Slack/PagerDuty/issues); the promote phase is a wiring point that reuses Lab 39's measurement and Lab 38's `derive_thresholds.py` rather than re-implementing them; real log capture, rate limiting, and PII handling are out of scope.

This completes Path 02's RAG track: from a single retriever to a routed, hardened, CI-gated, calibrated, self-monitoring system that tells you when to act.